In [3]:
import pandas as pd
import numpy as np

In [4]:
# Read in the data and combine them into one dataframe

df = pd.read_excel("data/raw/online_retail_II.xlsx", sheet_name=None)
df_merged = pd.concat([df['Year 2009-2010'], df['Year 2010-2011']], ignore_index=True)

df_merged.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [5]:
df_merged.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Data augmentation

First we split StockCode into three features for exploration. ProductCode will be the actual product identifier, Variation is the subtype given by a single character (color, style), and NonProductFlag contains abnormal items (postage, bank charges).

In [6]:
# Split the stockcode into two features
df_split = df_merged.copy()
df_split['ProductCode'] = df_split['StockCode'].astype(str).str.extract(r'(\d+)', expand=False)
df_split['Variation'] = df_split['StockCode'].astype(str).str.extract(r'^\d+(\D*)$', expand=False)
df_split['NonProductFlag'] = df_split['StockCode'].astype(str).str.extract(r'^(\D*)$', expand=False)
df_split.drop(['StockCode'], axis=1, inplace=True)

In [7]:
# Show the stock code an description for all unique variations
df_split.loc[df_split['Variation'].notna(), ['ProductCode', 'Variation', 'Description']].drop_duplicates()

,ProductCode,Variation,Description
0,85048,,15CM CHRISTMAS GLASS BALL 20 LIGHTS
1,79323,P,PINK CHERRY LIGHTS
2,79323,W,WHITE CHERRY LIGHTS
3,22041,,"RECORD FRAME 7"" SINGLE SIZE"
4,21232,,STRAWBERRY CERAMIC TRINKET BOX
...,...,...,...
1062371,23090,,missing
1063082,85123,A,CREAM HANGING HEART T-LIGHT HOLDER
1064015,85175,,NaN
1064386,23169,,smashed


We can look at the description for each NonProductFlag to confirm they are not actually purchases. After this, we will drop any item with no product code.

In [8]:
df_split.loc[df_split['NonProductFlag'].notna(), ['ProductCode', 'NonProductFlag', 'Description']].drop_duplicates()

,ProductCode,NonProductFlag,Description
89,NaN,POST,POSTAGE
735,NaN,D,Discount
2379,NaN,DOT,DOTCOM POSTAGE
2697,NaN,M,Manual
18410,NaN,BANK CHARGES,Bank Charges
62299,NaN,PADS,PADS TO MATCH ALL CUSHIONS
70975,NaN,ADJUST,Adjustment by john on 26/01/2010 16
71058,NaN,ADJUST,Adjustment by john on 26/01/2010 17
80231,NaN,DCGSSGIRL,NaN
83304,NaN,GIFT,NaN


## Dropping unneeded rows

- Anything with no product code is not a product, and should be removed.

- Anything with no customer ID cannot be used for the analysis, and should be removed.

- Rows with negative quantities are left, we will later take the sum of all transactions for each customer/product so leaving these should account for returns. For the same reason, rows with invoices starting with C (representing cancellations) are not removed. 

- Duplicated rows are removed. Some of these are due to overlap in the two sheets provided, some are likely due to input mistakes.

In [9]:
# Print missing values per column
print(df_split.isnull().sum())
print()
print("Rows with quantity = 0: ", (df_split['Quantity'] == 0).sum())
print("Rows with quantity < 0: ", (df_split['Quantity'] < 0).sum())
print()
print("Duplicated rows: ", df_split.duplicated().sum())

Invoice                 0
Description          4382
Quantity                0
InvoiceDate             0
Price                   0
Customer ID        243007
Country                 0
ProductCode          5579
Variation            6093
NonProductFlag    1061792
dtype: int64

Rows with quantity = 0:  0
Rows with quantity < 0:  22950

Duplicated rows:  34335


In [10]:
df_reduced = df_split.copy()

print(f"Initial number of rows: {df_reduced.shape[0]:,}")

# Remove anything without a product id
df_reduced = df_reduced.loc[df_reduced['ProductCode'].notna()]
print(f"Rows remaining after requiring product IDs: {df_reduced.shape[0]:,}")

# Remove anything without a customer id
df_reduced = df_reduced.loc[df_reduced['Customer ID'].notna()]
print(f"Rows remaining after requiring customer IDs: {df_reduced.shape[0]:,}")

# Remove duplicated rows
df_reduced = df_reduced.drop_duplicates()
print(f"Rows remaining after removing duplicates: {df_reduced.shape[0]:,}")

df_reduced.head()

Initial number of rows: 1,067,371
Rows remaining after requiring product IDs: 1,061,792
Rows remaining after requiring customer IDs: 820,906
Rows remaining after removing duplicates: 794,498


,Invoice,Description,Quantity,InvoiceDate,Price,Customer ID,Country,ProductCode,Variation,NonProductFlag
0,489434,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,85048,,NaN
1,489434,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,79323,P,NaN
2,489434,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,79323,W,NaN
3,489434,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,22041,,NaN
4,489434,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,21232,,NaN


In [11]:
# Make sure there are no cancellations with positive quantities (should be false)
print(((df_reduced['Invoice'].str.contains('C') == True) & (df_reduced['Quantity'] > 0)).any())

False


## Dropping unneeded features

- Description
    - Redundant, we can use product code to identify the product
- NonProductFlag
    - Generated solely to determine the meaning of the stockcode, not needed for analysis.
- Price, Country
    - Not relevant to the analysis

In [18]:
df_sliced = df_reduced.drop(columns=['Description', 'Price', 'Country', 'NonProductFlag'])

# Removing the full df from memory
# del df, df_merged

print(df_sliced.head())

  Invoice  Quantity         InvoiceDate  Customer ID ProductCode Variation
0  489434        12 2009-12-01 07:45:00      13085.0       85048          
1  489434        12 2009-12-01 07:45:00      13085.0       79323         P
2  489434        12 2009-12-01 07:45:00      13085.0       79323         W
3  489434        48 2009-12-01 07:45:00      13085.0       22041          
4  489434        24 2009-12-01 07:45:00      13085.0       21232          


## Redacting Customer IDs

We were asked to redact all customer IDs. We can do that by replacing each with a random new id.

In [19]:
rng = np.random.default_rng(42)

df_obscured = df_sliced.copy()
ids = df_obscured['Customer ID'].dropna().unique()
shuffled = rng.permutation(ids)

mapping = {cid: i for i, cid in enumerate(shuffled)}
df_obscured['AnonID'] = df_obscured['Customer ID'].map(mapping).astype('Int64')

df_obscured = df_obscured.drop(columns='Customer ID')

df_obscured.head()

,Invoice,Quantity,InvoiceDate,ProductCode,Variation,AnonID
0,489434,12,2009-12-01 07:45:00,85048,,4397
1,489434,12,2009-12-01 07:45:00,79323,P,4397
2,489434,12,2009-12-01 07:45:00,79323,W,4397
3,489434,48,2009-12-01 07:45:00,22041,,4397
4,489434,24,2009-12-01 07:45:00,21232,,4397


## Extracting the test section

All data from the most recent two months is removed. This will be used to test the results of the recommendation engine.

In [20]:
# get rows within the last two months

latestDateTime = df_obscured['InvoiceDate'].max()
cutoff = latestDateTime - pd.DateOffset(months=2)

train = df_obscured[df_obscured['InvoiceDate'] < cutoff]
test = df_obscured[df_obscured['InvoiceDate'] >= cutoff]

print(f"Train set size: {train.shape[0]:,}")
print(f"Test set size: {test.shape[0]:,}")

Train set size: 676,007
Test set size: 118,491


## Transforming the matrix

In [15]:
print("Unique Customers: ", train['AnonID'].unique().shape[0])
print("Unique Products: ", train['ProductCode'].unique().shape[0])
print("Unique Products (with variations): ", train['ProductCode'].str.cat([train['Variation']], sep=',').unique().shape[0])

Unique Customers:  5500
Unique Products:  3789
Unique Products (with variations):  4544


Including the product variations would add about 1000 extra products, which would make the recommendations less accurate. They would also be less useful, since it would very often recommend different variations of an item the person already bought.

In [21]:
train_simple = train.drop(columns=['Variation', 'InvoiceDate'])
test_simple = test.drop(columns=['Variation', 'InvoiceDate'])

train_simple.head()

,Invoice,Quantity,ProductCode,AnonID
0,489434,12,85048,4397
1,489434,12,79323,4397
2,489434,12,79323,4397
3,489434,48,22041,4397
4,489434,24,21232,4397


Using the raw quantities introduces bias from several sources. Some products are more expensive, so stores will buy fewer. This would mean the engine would prefer small cheap items that are often bought in bulk, even when they are less relevant. Also, large stores buy more items than small stores, so the engine would skew towards the preferences of the largest stores.

We could use a binary value representing purchased or not purchased, but that removes useful information. Buying something five times is a more significant indicator than buying it once. However, a raw "number of purchases" would skew towards older stores that have higher numbers for all products. Also, the difference between ordering once and twice is a more significant indicator than ordering 14 times compared to 15 times. To fit this, we can take the log of the purchase count.

In [48]:
# set n = purchase count
train_grouped = train_simple.groupby(['AnonID', 'ProductCode']).size().reset_index(name='n')
test_grouped = test_simple.groupby(['AnonID', 'ProductCode']).size().reset_index(name='n')

# validate
# print(train_simple.loc[(train_simple['AnonID'] == 4397) & (train_simple['ProductCode'] == '79323')])
# print()
# print(train_grouped[(train_grouped['AnonID'] == 4397) & (train_grouped['ProductCode'] == '79323')])

# log
train_grouped['n'] = np.log1p(train_grouped['n'])
test_grouped['n'] = np.log1p(test_grouped['n'])

In [50]:
# Reshape into customer x product
train_reshaped = train_grouped.pivot(index='AnonID', columns='ProductCode', values='n').fillna(0)
test_reshaped = test_grouped.pivot(index='AnonID', columns='ProductCode', values='n').fillna(0)

train_reshaped.head()

ProductCode,001,002,10002,1002,10080,10109,10120,10123,10124,10125,...,90203,90204,90205,90206,90208,90209,90210,90211,90212,90214
AnonID,,,,,,,,,,,,,,,,,,,,,
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [51]:
# Save the outputs
train_reshaped.to_csv("train.csv", index=False)
test_reshaped.to_csv("test.csv", index=False)

In [52]:
print(f"Train set shape: {train_reshaped.shape}")
print(f"Train set zero count: {train_reshaped.eq(0).sum().sum():,} ({(train_reshaped.eq(0).sum().sum() / (train_reshaped.shape[0] * train_reshaped.shape[1]) * 100):.2f}%)")
print(f"Train set memory usage: {train_reshaped.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
print(f"Test set memory usage: {test_reshaped.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

Train set shape: (5500, 3789)
Train set zero count: 20,435,460 (98.06%)
Train set memory usage: 159.04 MB
Test set memory usage: 50.00 MB
